### **Celda 1 - Importar librerías**

In [ ]:
import pandas as pd
import time

from geopy.geocoders import Nominatim, ArcGIS
from geopy.exc import GeocoderTimedOut, GeocoderServiceError
from geopy.extra.rate_limiter import RateLimiter

from tqdm.notebook import tqdm

### **Celda 2 - Funciones auxiliares**

In [ ]:
def obtener_coordenadas(direccion, geocod_nominatim, geocod_arcgis, max_intentos=3):
    """
    Busca coordenadas usando Nominatim y, si falla,
    intenta con ArcGIS.
    """

    # Intentar con Nominatim
    for intento in range(max_intentos):
        try:
            location = geocod_nominatim(direccion)

            if location:
                return location.latitude, location.longitude
            else:
                break

        except (GeocoderTimedOut, GeocoderServiceError) as e:
            espera = 2 ** intento * 2
            print(f"Nominatim error: {e}")
            time.sleep(espera)

    # Fallback ArcGIS
    for intento in range(max_intentos):
        try:
            location = geocod_arcgis(direccion)

            if location:
                return location.latitude, location.longitude
            else:
                break

        except (GeocoderTimedOut, GeocoderServiceError) as e:
            espera = 2 ** intento * 2
            print(f"ArcGIS error: {e}")
            time.sleep(espera)

    return None, None

def leer_csv_con_codificacion(archivo):
    for enc in ['utf-8', 'latin-1', 'cp1252', 'ISO-8859-1']:
        try:
            return pd.read_csv(archivo, encoding=enc)
        except UnicodeDecodeError:
            continue

    raise ValueError(
        "No se pudo leer el archivo con ninguna codificación común."
    )

### **Celda 3 - Configuración del archivo.**

In [ ]:
archivo_entrada = "inmuebles_limpios.csv"
archivo_salida = "inmuebles_geolocalizados.csv"

# None = continuar automáticamente
# 5000 = comenzar desde la fila 5000
inicio_manual = None

### **Celda 4 - Cargar archivo CSV**

In [ ]:
import os

df = leer_csv_con_codificacion(archivo_entrada)

total = len(df)

if "direccion" not in df.columns:
    raise ValueError(
        "No se encontró la columna 'direccion' en el archivo."
    )

if inicio_manual is not None:

    start_idx = inicio_manual - 1

    print(f"Iniciando manualmente desde fila {inicio_manual}")

elif os.path.exists(archivo_salida):

    try:
        df_existente = pd.read_csv(archivo_salida)
        start_idx = len(df_existente)

    except Exception:
        start_idx = 0

    print(
        f"Archivo existente encontrado.\n"
        f"Continuando desde fila {start_idx + 1}"
    )

else:

    start_idx = 0

    print("No existe archivo de salida.")
    print("Comenzando desde el inicio.")

if start_idx >= total:
    print("No hay registros pendientes por procesar.")
    raise SystemExit

df_pendiente = df.iloc[start_idx:].copy()

print(f"Total registros: {total}")
print(f"Pendientes por procesar: {len(df_pendiente)}")

### **Celda 5 - Configurar geocodificadores**

In [ ]:
nominatim = Nominatim(
    user_agent="geocodificador_inmuebles",
    timeout=30
)

arcgis = ArcGIS(timeout=30)

geocod_nominatim = RateLimiter(
    nominatim.geocode,
    min_delay_seconds=1.1,
    max_retries=2,
    error_wait_seconds=5
)

geocod_arcgis = RateLimiter(
    arcgis.geocode,
    min_delay_seconds=0.5,
    max_retries=2,
    error_wait_seconds=5
)

print("Geocodificadores listos.")

### **Celda 6 - Obtener coordenadas**

In [ ]:
import os

file_exists = os.path.exists(archivo_salida)

with open(archivo_salida, "a", encoding="utf-8") as f:

    if not file_exists:

        columnas = list(df.columns)
        columnas.remove("direccion")

        encabezado = ",".join(columnas)
        encabezado += ",latitud,longitud\n"

        f.write(encabezado)

    for _, row in tqdm(
        df_pendiente.iterrows(),
        total=len(df_pendiente),
        desc="Geolocalizando"
    ):

        direccion = row["direccion"]

        if pd.isna(direccion) or str(direccion).strip() == "":

            lat = None
            lon = None

        else:

            direccion_busqueda = f"{direccion}, México"

            lat, lon = obtener_coordenadas(
                direccion_busqueda,
                geocod_nominatim,
                geocod_arcgis
            )

        fila_salida = row.drop("direccion").copy()

        fila_salida["latitud"] = lat
        fila_salida["longitud"] = lon

        pd.DataFrame([fila_salida]).to_csv(
            f,
            header=False,
            index=False
        )

        f.flush()

### **Celda 7 - Guardar datos**

In [ ]:
resultado = pd.read_csv(archivo_salida)

print("Registros guardados:", len(resultado))
print(
    "Con coordenadas:",
    resultado["latitud"].notna().sum()
)

resultado.head()